<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Thresholds and calibration

**[Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/)** · Machine Learning: From Problem to Reliable Model · Module 6, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** choose the threshold on the validation set for the senior team's capacity, check whether the scores can be read as chances (calibration, the reliability chart and the Brier score), look at the test set once with `evaluate.py`, and finish Module 6 with its practice and its check.

| | |
|---|---|
| **Time** | About 75 minutes, with the module practice |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | The precision–recall trade-off, from [The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/). Tuning and the one-standard-error rule, from [Tune hyperparameters](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/#the-one-standard-error-rule). |
| **You do not need** | The local project. The setup below downloads the data, `ticket_model.py` and `train.py`. You write `evaluate.py` in section 8. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M06-L03-thresholds-and-calibration/thresholds-and-calibration-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py`, `train.py` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. `ticket_model.py` is the final 11-feature version. `train.py` trains the model and chooses the threshold; you write it yourself in [Package artifacts](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/). Section 8 runs it.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py", "train.py"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: the experiment log

The next cell defines two helpers for your **experiment log**, the file `experiment_log.csv` in `ticket-model/`. You start the log in [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log).

- `log_experiment(id, change, cv_ap, cv_std, valid_ap, decision, reason)` adds one row, with today's date. If the id is already in the log, it replaces that row, so you can run a cell again.
- `show_log()` shows the log as a table.

It also writes the 14 rows that you added in the earlier lessons (E01 to E14), so this notebook works without them. If you keep your own log in the local project, skip the last line of the cell.

> **Check.** You should see the log with 14 rows, from E01 to E14.

In [ ]:
# The experiment log: one row per experiment, in experiment_log.csv.
import datetime
from pathlib import Path

LOG_FILE = Path("experiment_log.csv")
LOG_COLUMNS = ["id", "date", "change", "cv_ap", "cv_std", "valid_ap", "decision", "reason"]


def log_experiment(id, change, cv_ap, cv_std, valid_ap=None, decision="", reason=""):
    """Add one row to the log (or replace the row with the same id)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}) if LOG_FILE.exists() else pd.DataFrame(columns=LOG_COLUMNS)
    row = {"id": id, "date": datetime.date.today().isoformat(), "change": change,
           "cv_ap": round(float(cv_ap), 3), "cv_std": round(float(cv_std), 3),
           "valid_ap": None if valid_ap is None else round(float(valid_ap), 3),
           "decision": decision, "reason": reason}
    log = pd.concat([log[log["id"] != id], pd.DataFrame([row])], ignore_index=True)
    log.sort_values("id").to_csv(LOG_FILE, index=False)


def show_log(columns=("id", "change", "cv_ap", "cv_std", "valid_ap", "decision")):
    """The log as a table (without the date and the reason, unless you ask for them)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}).sort_values("id")
    with pd.option_context("display.max_colwidth", 60, "display.width", 200):
        print(log[list(columns)].fillna("").to_string(index=False))


# The rows of the earlier lessons (id, change, cv_ap, cv_std, valid_ap, decision, reason).
EARLIER_ROWS = [
    ('E01', 'Baseline: one score for every ticket (the train rate)', 0.117, 0.000, 0.155, 'baseline', 'The floor: any useful model must be above it.'),
    ('E02', "Baseline: the rule 'flag priority 1', as a 0/1 score", 0.178, 0.008, 0.228, 'baseline', 'What Larkfield does today: the model must beat it.'),
    ('E03', 'Logistic regression, the 10 C04 features', 0.327, 0.012, 0.353, 'keep', 'Far above both baselines; train AP 0.328, so it does not overfit.'),
    ('E04', "E03 with class_weight='balanced'", 0.323, 0.012, None, 'reject', 'Lower than E03 on all 5 folds; one more setting for no gain.'),
    ('E05', "Baseline: the rule 'flag priority 1 or team payment'", 0.172, 0.006, None, 'reject', 'Lower than the simpler rule E02.'),
    ('E06', 'Decision tree, max_depth=3', 0.234, 0.011, None, 'reject', 'Far below E03.'),
    ('E07', 'Decision tree, max_depth=6', 0.258, 0.010, None, 'reject', 'The best of the three trees, still far below E03.'),
    ('E08', 'Decision tree, no depth limit', 0.137, 0.004, None, 'reject', 'It memorises train (train AP 1.000) and fails on new tickets.'),
    ('E09', 'Decision tree, no depth limit, min_samples_leaf=100', 0.278, 0.017, None, 'reject', 'The best tree so far, still far below E03.'),
    ('E10', 'E03 with C=0.0001 (a very strong penalty)', 0.302, 0.012, None, 'reject', 'It underfits: both train and CV AP fall.'),
    ('E11', 'Random forest, 200 trees, default settings', 0.290, 0.014, None, 'reject', 'Its trees memorise (train AP 1.000); below E03.'),
    ('E12', 'Random forest, 200 trees, min_samples_leaf=20', 0.325, 0.018, 0.382, 'reject', 'A tie with E03 (higher on 3 folds of 5), and 2,000 times bigger as a file.'),
    ('E13', 'Gradient boosting, default settings', 0.316, 0.010, 0.369, 'reject', 'Lower than E03 on all 5 folds.'),
    ('E14', 'Gradient boosting, max_depth=3, learning_rate=0.05', 0.327, 0.016, 0.374, 'reject', 'A tie with E03; more complex and harder to explain.'),
]
for row in EARLIER_ROWS:
    log_experiment(*row)
show_log()

The next cell adds the rows of the earlier lessons of Module 6 (E15 to E21). If you keep your own log in the local project, skip this cell.

> **Check.** You should see the log with 21 rows, from E01 to E21.

In [ ]:
# The rows of the earlier lessons of Module 6 (id, change, cv_ap, cv_std, valid_ap, decision, reason).
M06_ROWS = [
    ('E15', 'E03 + prior_escalations_90d (11 features)', 0.339, 0.016, 0.391, 'keep', 'Known at creation, and higher than E03 on all 5 folds.'),
    ('E16', 'E03 + order_value_band', 0.328, 0.014, 0.358, 'reject', 'It repeats order_value; the gain is smaller than the noise.'),
    ('E17', 'E03 + customer_escalation_rate', 0.445, 0.015, 0.485, 'reject', 'Leak: computed over later targets, so not known at creation.'),
    ('E18', 'E03 + priority_now', 0.851, 0.024, 0.873, 'reject', 'Leak: the help desk sets it to 1 when a ticket escalates.'),
    ('E19', 'Grid search best: gradient boosting, learning_rate=0.1, max_depth=2', 0.342, 0.018, 0.395, 'reject', 'Within one standard error (0.008) of E15, and more complex.'),
    ('E20', 'E15 with C=10 (the best C; C=0.01 to 10 are within 0.002)', 0.339, 0.016, None, 'reject', 'No real difference from C=1, so keep the default.'),
    ('E21', 'Random search best: gradient boosting, learning_rate=0.208, max_depth=2, min_samples_leaf=20', 0.344, 0.017, None, 'reject', 'Less than one standard error (0.0077) above E15.'),
]
for row in M06_ROWS:
    log_experiment(*row)
show_log()

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A threshold for a capacity, on ten tickets

A **threshold** turns a score into a flag: flag the ticket when its score is at least the threshold. In Module 3 you saw the trade-off: a lower threshold flags more tickets, so recall goes up and precision goes down. Grace's team can review at most a share of the tickets, the **capacity**. A lower threshold catches more escalations, so the best threshold is the **lowest** one that stays within the capacity.

`choose_threshold()` does this in steps of 0.01. It is the same function as in the course's `train.py`. Run the cell.

In [ ]:
def choose_threshold(scores, capacity):
    """The lowest threshold (in steps of 0.01) that flags no more than
    `capacity` of the tickets. Lower thresholds flag more tickets and catch
    more escalations, so the lowest one that fits catches the most."""
    for step in range(1, 100):
        threshold = step / 100
        if (scores >= threshold).mean() <= capacity:
            return threshold
    return 1.0

Here are the scores of ten tickets. The capacity is 3 of 10 (30%).

> **Predict.** Which threshold will the function return? Count the scores at or above 0.43, then at or above 0.44. Then run the cell.

In [ ]:
scores_10 = np.array([0.81, 0.62, 0.55, 0.43, 0.38, 0.30, 0.22, 0.15, 0.09, 0.04])
threshold_10 = choose_threshold(scores_10, 0.3)
print("threshold:", threshold_10, "| flagged:", (scores_10 >= threshold_10).sum())

> **Check.** You should see `threshold: 0.44 | flagged: 3`. At 0.43, four tickets (40%) are flagged: too many.

## 2. The threshold on the real validation set

The next cell fits the final model on train and scores the May tickets. Then it prints the four counts at six thresholds. Run it.

In [ ]:
from sklearn.metrics import confusion_matrix
from ticket_model import FEATURES, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])
scores = pipeline.predict_proba(valid[FEATURES])[:, 1]
for threshold in [0.5, 0.3, 0.2, 0.19, 0.15, 0.1]:
    flags = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(valid[TARGET], flags).ravel()
    print(f"{threshold:4}: flagged {tp + fp:3d} ({(tp + fp) / len(valid):5.1%})  TP {tp:3d}  FP {fp:3d}  FN {fn:3d}  TN {tn:3d}"
          f"  precision {tp / (tp + fp):.3f}  recall {tp / (tp + fn):.3f}")

> **Check.** You should see six lines. At 0.5: flagged 25 (2.1%), recall 0.093. At 0.19: flagged 227 (19.3%), TP 86, FP 141, FN 97, TN 855, precision 0.379, recall 0.470. At 0.1: flagged 475 (40.3%), recall 0.699.

> **Predict.** The capacity is 20% of tickets. Which threshold will `choose_threshold()` return? Then run the cell.

In [ ]:
threshold = choose_threshold(scores, 0.20)
flags = (scores >= threshold).astype(int)
print("threshold:", threshold, f"| flagged {flags.sum()} of {len(valid)} ({flags.mean():.1%})")

> **Check.** You should see `threshold: 0.19 | flagged 227 of 1179 (19.3%)`. At 0.18, the model flags 248 tickets (21.0%): over the capacity.

The model catches 86 of the 183 escalations (recall 0.470) with 227 reviews. The priority-1 rule catches 61 with 163 reviews (recall 0.333). The improvement target (at most 20% flagged, at least 40% caught) is met on valid.

> **Your turn.** Grace says that in a busy week her team can review only 15% of tickets. Compute `threshold_15` with `choose_threshold()`, and `flagged_15`, the number of May tickets it flags. Run the check cell.

In [ ]:
threshold_15 = choose_threshold(scores, 0.15)
flagged_15 = int((scores >= threshold_15).sum())
print(threshold_15, flagged_15)

In [ ]:
expect_hash('(threshold_15, flagged_15)', (threshold_15, flagged_15), '5e5437c1be48b77e')

## 3. Failure case: the default threshold of predict()

Tomás uses `pipeline.predict()`, as in most tutorials. Run the cell.

In [ ]:
flags_default = pipeline.predict(valid[FEATURES])
print("predict() flags:", flags_default.sum(), f"({flags_default.mean():.1%})")

> **Check.** You should see `predict() flags: 25 (2.1%)`.

`predict()` uses a threshold of 0.5. Only 25 tickets have a score that high, so the senior team gets 25 reviews and misses 166 of 183 escalations. No error, no warning. Use `predict_proba()` and your own threshold.

## 4. Calibration: can a score be read as a chance?

A weather forecaster who says "70% chance of rain" is good if it rains on about 7 of every 10 such days. A model is **calibrated** in the same way when its scores match how often things happen: of the tickets with a score near 0.25, about 25% escalate. [Calibration and slices](https://learning.nextia-ai.com/courses/math/m06/calibration-and-slices/) in the mathematics course explains the idea.

The **Brier score** measures it in one number: the mean of (score − outcome)², where the outcome is 1 or 0. Lower is better. Here are four tickets.

> **Predict.** Which ticket adds the most to the Brier score? Then run the cell.

In [ ]:
scores_4 = np.array([0.9, 0.2, 0.6, 0.1])
outcomes_4 = np.array([1, 0, 0, 0])
print("squared errors:", ((scores_4 - outcomes_4) ** 2).round(2))
print("Brier score:", round(float(((scores_4 - outcomes_4) ** 2).mean()), 4))

> **Check.** You should see `squared errors: [0.01 0.04 0.36 0.01]` and `Brier score: 0.105`. The third ticket (score 0.6, not escalated) adds the most.

Now group the real scores into bins, and compare the mean score with the observed escalation rate in each bin, on train and on valid. Run the cell.

In [ ]:
train_scores = pipeline.predict_proba(train[FEATURES])[:, 1]
bins = [0, 0.05, 0.1, 0.2, 0.3, 0.5, 1]

def reliability(part, part_scores):
    """Tickets, mean score and observed escalation rate per score bin."""
    table = pd.DataFrame({"score": part_scores, "escalated": part[TARGET].to_numpy()})
    return table.groupby(pd.cut(part_scores, bins), observed=True).agg(
        tickets=("escalated", "size"), mean_score=("score", "mean"), rate=("escalated", "mean")).round(3)

print("train\n", reliability(train, train_scores))
print("valid\n", reliability(valid, scores))

> **Check.** On train, each mean score is close to its rate: for example 0.244 and 0.249 in the bin 0.2–0.3. On valid, every rate is higher than the mean score: 0.241 and 0.315 in the same bin.

The chart shows the same bins. A dot on the dashed diagonal means "score = observed rate". Run the cell.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 0.7], [0, 0.7], linestyle="--", color="grey", label="perfect calibration")
for name, part, part_scores in [("train", train, train_scores), ("valid (May)", valid, scores)]:
    table = reliability(part, part_scores)
    ax.plot(table["mean_score"], table["rate"], marker="o", label=name)
ax.set_xlabel("Mean score in the bin")
ax.set_ylabel("Observed escalation rate in the bin")
ax.legend()
plt.show()

> **Check.** The train line lies close to the diagonal (only the last bin, 289 tickets, is a little below it). The valid line lies above the diagonal in every bin: the model **under-estimates** the risk in May.

> **Your turn.** Compute the Brier score of the model on valid by hand, with numpy: the mean of (score − outcome)². Round it to 4 decimals and put it in `brier_valid`. Run the check cell.

In [ ]:
brier_valid = round(float(np.mean((scores - valid[TARGET].to_numpy()) ** 2)), 4)
print(brier_valid)

In [ ]:
expect_hash('brier_valid', brier_valid, '3e6b0ec64b51c9eb')

Compare it with `brier_score_loss` and with a constant score. Run the cell.

In [ ]:
from sklearn.metrics import brier_score_loss

print(f"train: model {brier_score_loss(train[TARGET], train_scores):.4f}, constant 0.117 {brier_score_loss(train[TARGET], np.full(len(train), 0.117)):.4f}")
print(f"valid: model {brier_score_loss(valid[TARGET], scores):.4f}, constant 0.117 {brier_score_loss(valid[TARGET], np.full(len(valid), 0.117)):.4f}")
print(f"valid: mean score {scores.mean():.3f}, escalation rate {valid[TARGET].mean():.3f}")

> **Check.** You should see `train: model 0.0902, constant 0.117 0.1034`, `valid: model 0.1155, constant 0.117 0.1326`, and `mean score 0.120, escalation rate 0.155`.

The Brier score is worse on valid (0.1155 against 0.0902). Part of that is normal: escalations are more common in May, and a 15% rate gives a larger Brier score than a 12% rate even for a perfect model. The rest is the gap: the model gives 12.0% on average, and 15.5% escalate.

## 5. Where the gap is: warranty tickets

The new warranty policy of 2026-05-01 sends more warranty questions to the senior team. Compare the mean score and the rate by team. Run the cell.

In [ ]:
by_team = valid.assign(score=scores).groupby("team").agg(
    tickets=(TARGET, "size"), mean_score=("score", "mean"), rate=(TARGET, "mean"))
print(by_team.round(3))

> **Check.** For `warranty`, you should see 168 tickets, mean score 0.163 and rate 0.286. The other teams have smaller gaps (for example payment: 0.214 and 0.233).

The model learned the warranty rate from before the policy. Its warranty scores are too low by about 12 percentage points.

## 6. Ranking still works

The capacity rule uses only the **order** of the scores: it flags the top 20%. A score that is too low for everyone keeps the same order. Test it: add 0.035 to every score (the mean moves from 0.120 to about 0.155) and compute the average precision again. Run the cell.

In [ ]:
from sklearn.metrics import average_precision_score

shifted = np.clip(scores + 0.035, 0, 1)
print(f"mean score: {scores.mean():.3f} -> {shifted.mean():.3f}")
print("average precision:", round(average_precision_score(valid[TARGET], scores), 3), "->",
      round(average_precision_score(valid[TARGET], shifted), 3))

> **Check.** You should see `mean score: 0.120 -> 0.155` and `average precision: 0.391 -> 0.391`.

The order did not change, so the average precision did not change. **The threshold of 0.19 still flags the riskiest 19.3% of May tickets.** But the scores must not be shown to people as "the chance of escalation": a warranty ticket with a score of 0.16 escalates almost twice as often. The fix is new data, not a new threshold: recalibrate on recent labelled tickets, or retrain with May and June (Module 8).

## 7. Optional: a cost view

Grace's rough costs: one review costs about 10 minutes of a senior agent. Say a missed escalation costs about 60 minutes (later handling, and an unhappy customer). Then the total cost at a threshold is 10 × flagged + 60 × missed. This view is optional. The capacity is still the limit.

In [ ]:
truth = valid[TARGET].to_numpy()
costs = {}
for step in range(0, 101):
    t = step / 100
    costs[t] = 10 * (scores >= t).sum() + 60 * ((scores < t) & (truth == 1)).sum()
cheapest = min(costs, key=costs.get)
print(f"cheapest threshold {cheapest}: {costs[cheapest]} minutes, flags {(scores >= cheapest).mean():.1%}")
print(f"threshold 0.19: {costs[0.19]} minutes")

> **Check.** You should see `cheapest threshold 0.14: 7470 minutes, flags 29.3%` and `threshold 0.19: 8090 minutes`.

With these costs, more reviews would pay for themselves, but 29.3% is above the capacity. The team cannot do them. That is a message for Grace: more senior time would catch more escalations. The threshold stays at 0.19 until the capacity changes.

## 8. Test once with evaluate.py

Every choice is made: the features (Lesson 1), the settings (Lesson 2) and the threshold (this lesson). Now, and only now, look at the **test set**: the June tickets that no step has used. It is the sealed exam of Module 3: you take it once, and you do not change your answers after you see the score.

Run the next cell. It writes `evaluate.py`, the course's file.

In [ ]:
%%writefile evaluate.py
"""Report the saved model's results on the test set, once.

Nextia Learning, C05, Module 4 (Validation strategy) and Module 6.

    python evaluate.py

Uses the threshold that train.py chose on valid. Writes
reports/test_report.json. If the report exists already, it stops: look at
the test set once, and do not change the model after you look.
"""

import json
import sys
from pathlib import Path

import joblib
from sklearn.metrics import average_precision_score, brier_score_loss, confusion_matrix, roc_auc_score

from ticket_model import FEATURES, TARGET, load

REPORT = Path("reports/test_report.json")


def main():
    if REPORT.exists() and "--again" not in sys.argv:
        sys.exit(f"{REPORT} exists: the test set was used already. Read the report. "
                 "Run with --again only to reproduce it, never to choose a change.")
    pipeline = joblib.load("model/escalation_model.joblib")
    info = json.loads(Path("model/model_info.json").read_text(encoding="utf-8"))
    test = load("test")
    scores = pipeline.predict_proba(test[FEATURES])[:, 1]
    flags = (scores >= info["threshold"]).astype(int)
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(test[TARGET], flags, labels=[0, 1]).ravel())
    rule = (test["priority"] == 1).astype(int)
    rule_tp = int((rule & test[TARGET]).sum())
    report = {
        "rows": len(test), "escalated": int(test[TARGET].sum()), "threshold": info["threshold"],
        "roc_auc": round(roc_auc_score(test[TARGET], scores), 3),
        "average_precision": round(average_precision_score(test[TARGET], scores), 3),
        "brier": round(brier_score_loss(test[TARGET], scores), 4),
        "mean_score": round(float(scores.mean()), 3), "escalated_rate": round(float(test[TARGET].mean()), 3),
        "confusion": {"tp": tp, "fp": fp, "fn": fn, "tn": tn},
        "flag_rate": round((tp + fp) / len(test), 3),
        "precision": round(tp / (tp + fp), 3), "recall": round(tp / (tp + fn), 3),
        "baseline_priority_1": {"flagged": int(rule.sum()), "flag_rate": round(float(rule.mean()), 3),
                                "precision": round(rule_tp / rule.sum(), 3),
                                "recall": round(rule_tp / test[TARGET].sum(), 3)},
    }
    REPORT.parent.mkdir(exist_ok=True)
    REPORT.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

> **Check.** You should see `Writing evaluate.py` (or `Overwriting`).

`evaluate.py` needs the saved model. `train.py` fits the pipeline on train, chooses the threshold on valid with `choose_threshold()`, and saves the model and the threshold in `model/`. `!"{sys.executable}"` runs a script with the notebook's own Python, like `python train.py` in a terminal. Run the cell.

In [ ]:
import sys
!"{sys.executable}" train.py

> **Check.** The first line is `Trained on 16939 tickets. Threshold 0.19 for a review capacity of 20%.` The second line shows the valid results: ROC AUC 0.752, average precision 0.391, flagged 227 of 1179 (19.3%), precision 0.379, recall 0.47.

> **Warning.** The next cell uses the test set. Run it once. Read the result, and do not change the model, the features or the threshold after it.

In [ ]:
!"{sys.executable}" evaluate.py

> **Check.** You should see a JSON report with `"roc_auc": 0.749`, `"average_precision": 0.345`, `"brier": 0.1207`, the confusion counts `"tp": 70, "fp": 126, "fn": 87, "tn": 719`, `"flag_rate": 0.196`, `"precision": 0.357`, `"recall": 0.446`, and the rule's `"precision": 0.367, "recall": 0.35`.

If you run this notebook a second time in the same folder, the cell above prints the guard message instead. That is correct: the report from your first run is in `reports/test_report.json`.

Now run `evaluate.py` again, as Tomás would after "one small change".

In [ ]:
!"{sys.executable}" evaluate.py

> **Check.** You should see `reports/test_report.json exists: the test set was used already. Read the report. Run with --again only to reproduce it, never to choose a change.`

The guard is a reminder, not a lock: `--again` reproduces the same report. Read the report from the file. Run the cell.

In [ ]:
import json
from pathlib import Path

report = json.loads(Path("reports/test_report.json").read_text())
print({key: report[key] for key in ["roc_auc", "average_precision", "flag_rate", "precision", "recall"]})

In [ ]:
expect('the test average precision', report["average_precision"], 0.345)

The test AP (0.345) is lower than valid (0.391). That is normal: valid helped to choose, so it is a little optimistic, and June is a new month. The model still meets the improvement target on test: 19.6% flagged, recall 0.446 (the rule: 0.350). **We do not change anything after we look.**

## 9. The experiment log at the end of Module 6

The log records **experiments**: tries that compete to become the model, each with its CV score. The threshold and the test choose no model, so they get no row. The threshold is saved with the model in `model/model_info.json`, and the test result in `reports/test_report.json`. The log ends with E15 as the model to keep. Run the cell to see it.

In [ ]:
show_log()

> **Check.** You should see 21 rows. E15 is the last `keep`.

A new row needs new data, for example a retrain with May and June (Module 8).

## 10. Module practice: tune and choose a threshold for 25%

Grace may add a part-time senior agent. Then the capacity is 25%. Do the whole workflow again, without the test set:

1. Tune a small search space with CV on train.
2. Choose with the one-standard-error rule.
3. Choose the threshold on valid for 25%.

Run the search. It has four settings: `C` 0.1 or 1, and `class_weight` `None` or `"balanced"` (which gives escalated tickets more weight in the fit).

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(5, shuffle=True, random_state=0)
small_grid = {"model__C": [0.1, 1], "model__class_weight": [None, "balanced"]}
search = GridSearchCV(build_pipeline(), small_grid, cv=cv, scoring="average_precision")
search.fit(train[FEATURES], train[TARGET])
results = pd.DataFrame(search.cv_results_)
results["se"] = results["std_test_score"] / 5 ** 0.5
print(results[["params", "mean_test_score", "std_test_score", "se", "rank_test_score"]].round(4).to_string(index=False))

> **Check.** You should see four rows. The best is `C` 1 with `class_weight` `None`: 0.3393, standard error 0.0070. The lowest is 0.3364.

All four are within one standard error of the best (0.3393 − 0.0070 = 0.3323). Keep the simplest: the default model, `C` = 1 without class weights. It is the pipeline that you already have.

> **Your turn.** Choose the threshold for a capacity of 25% on valid with the default model's `scores`. Put it in `threshold_25`, and the number of flagged May tickets in `flagged_25`. Run the check cell.

In [ ]:
threshold_25 = choose_threshold(scores, 0.25)
flagged_25 = int((scores >= threshold_25).sum())
print(threshold_25, flagged_25, f"{flagged_25 / len(valid):.1%}")

In [ ]:
expect_hash('(threshold_25, flagged_25)', (threshold_25, flagged_25), 'b2e61224de9d5ff8')

Now compare with the balanced model. Its scores are much higher, so its threshold is different. Run the cell.

In [ ]:
from sklearn.linear_model import LogisticRegression

balanced = build_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced")).fit(train[FEATURES], train[TARGET])
balanced_scores = balanced.predict_proba(valid[FEATURES])[:, 1]
for name, s in [("default", scores), ("balanced", balanced_scores)]:
    t = choose_threshold(s, 0.25)
    f = s >= t
    print(f"{name:8s} mean score {s.mean():.3f}  threshold {t}  flagged {f.sum()} ({f.mean():.1%})"
          f"  recall {(f & (truth == 1)).sum() / truth.sum():.3f}")

> **Check.** You should see the default model at threshold 0.16 (291 flagged, 24.7%, recall 0.557) and the balanced model at threshold 0.59 (287 flagged, 24.3%, recall 0.552).

Almost the same tickets, almost the same recall, at very different thresholds. A threshold belongs to one model: if the model changes, choose the threshold again on valid. The balanced model's mean score (0.415) is far from the rate (0.155), so its scores are not chances at all.

## 11. Module check: report the held-out results

The test report from section 8 is the final held-out result. The next cell reads `reports/test_report.json` again. If this folder has no report (for example, after a new Colab session), it runs `train.py` and `evaluate.py` once to make it. Run it.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

if not Path("reports/test_report.json").exists():
    subprocess.run([sys.executable, "train.py"], check=True)
    subprocess.run([sys.executable, "evaluate.py"], check=True, stdout=subprocess.DEVNULL)
report = json.loads(Path("reports/test_report.json").read_text())
print({key: report[key] for key in ["threshold", "roc_auc", "average_precision", "brier", "flag_rate", "precision", "recall"]})
print("rule:", report["baseline_priority_1"])

> **Check.** You should see threshold 0.19, ROC AUC 0.749, average precision 0.345, Brier 0.1207, flag rate 0.196, precision 0.357 and recall 0.446, and the rule's flag rate 0.15, precision 0.367 and recall 0.35.

Write the report in four or five sentences: the model, the threshold and where it was chosen, the test results next to the rule, and the limits. Do **not** use these numbers for another choice: not a new threshold for 25%, not another model. The lesson page has Priya's report and the module check.

## Next

Module 6 is complete. Module 7 looks at the errors: which tickets the model misses, for whom it works less well, and when a simple model is the better choice. The [lesson page](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/) has the explorer, the reliability chart and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Slice analysis](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.